# M9 Forecast Training — Colab TPU

Trains **AvgProcCost** and **TPV** forecast artifacts for the `ml-service` container.

**Runtime → Change runtime type → TPU v2-8**

## Outputs
- `m9_artifacts.zip` — cost forecast artifacts (`m9/{mcc}/{ctx_len}/`)
- `tpv_artifacts.zip` — TPV forecast artifacts (`tpv/{mcc}/{ctx_len}/`)

Copy the artifact folders into `ml_service/artifacts/m9/` and `ml_service/artifacts/tpv/` respectively.

In [ ]:
# Cell 1: Install dependencies
!pip install -q numpy pandas scikit-learn joblib

In [ ]:
# Cell 2: Upload processed_transactions_4mcc.csv
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import files

print("Click the button below to upload processed_transactions_4mcc.csv")
print("(or any monthly v2 CSV with the required columns)")
print()

uploaded = files.upload()

UPLOAD_FILENAME = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {UPLOAD_FILENAME} ({len(uploaded[UPLOAD_FILENAME]):,} bytes)")

In [ ]:
# Cell 3: Configuration
import os
from pathlib import Path

# ── User config ──────────────────────────────────────────────────────────
DATA_PATH = Path(UPLOAD_FILENAME)
MCC = 5411
WINDOW_YEARS = 3

# ── Pipeline constants (shared by both services) ─────────────────────────
SUPPORTED_CONTEXT_LENS = [1, 3, 6]
HORIZON_LEN = 3
TARGET_COV = 0.90
MIN_POOL = 10
KNN_K = 10
_VOL_EPS = 1e-6

# Stratification
VOL_MIN_GAIN_ABS = 0.05
VOL_MIN_GAIN_REL = 0.01
VOL_TEST_COV_SLACK = 0.03
VOL_BUCKET_SCHEMES = {
    "low-mid-high_50_85":          [0.00, 0.50, 0.85, 1.00],
    "low-mid-high_40_80":          [0.00, 0.40, 0.80, 1.00],
    "low-mid-high_60_90":          [0.00, 0.60, 0.90, 1.00],
    "low-mid-high-vhigh_50_75_90": [0.00, 0.50, 0.75, 0.90, 1.00],
    "low-mid-high-vhigh_40_70_88": [0.00, 0.40, 0.70, 0.88, 1.00],
    "low-mid-high-vhigh_60_85_95": [0.00, 0.60, 0.85, 0.95, 1.00],
}

# GBR hyper-parameters
GBR_N_ESTIMATORS = 120
GBR_LEARNING_RATE = 0.05
GBR_MAX_DEPTH = 2
GBR_MIN_SAMPLES_LEAF = max(20, MIN_POOL)
GBR_SUBSAMPLE = 0.8
GBR_RANDOM_STATE = 4121

# Cost-type columns
COST_TYPE_COLS = [f"cost_type_{i}_pct" for i in range(1, 62)]

# v2 required columns — cost
COST_V2_REQUIRED = [
    "std_proc_cost_pct", "iqr_proc_cost_pct", "std_txn_amount",
    "median_txn_amount", "n_unique_cost_types", "median_proc_cost_pct",
]
# v2 required columns — TPV
TPV_V2_REQUIRED = [
    "std_txn_amount", "median_txn_amount", "avg_transaction_value",
    "transaction_count", "total_processing_value",
]

# TPV target columns
TARGET_COL = "total_processing_value"
LOG_TARGET = "log_tpv"

# Artifact output paths
M9_ARTIFACTS = Path("artifacts/m9")
TPV_ARTIFACTS = Path("artifacts/tpv")

print(f"MCC={MCC}  WINDOW={WINDOW_YEARS}yr  DATA={DATA_PATH}")
print(f"Context lens: {SUPPORTED_CONTEXT_LENS}")

In [ ]:
# Cell 4: Shared helper functions
from __future__ import annotations

import json
import math
import tempfile
import time
from collections import defaultdict
from datetime import date, datetime, timezone
from typing import Dict, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import HuberRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


def build_scenarios(df: pd.DataFrame, context_len: int) -> List[dict]:
    """Slide a (context_len, HORIZON_LEN) window over each merchant's history."""
    scenarios = []
    for mid, mdf in df.groupby("merchant_id"):
        mdf = mdf.sort_values(["year", "month"]).reset_index(drop=True)
        n = len(mdf)
        for i in range(n - context_len - HORIZON_LEN + 1):
            ctx = mdf.iloc[i : i + context_len].copy().reset_index(drop=True)
            hor = mdf.iloc[i + context_len : i + context_len + HORIZON_LEN].copy().reset_index(drop=True)
            ctx_span = (
                (ctx.iloc[-1]["year"] - ctx.iloc[0]["year"]) * 12
                + ctx.iloc[-1]["month"] - ctx.iloc[0]["month"]
            )
            if ctx_span != context_len - 1:
                continue
            ctx_end_yr = int(ctx.iloc[-1]["year"])
            ctx_end_mo = int(ctx.iloc[-1]["month"])
            exp_yr = ctx_end_yr + (1 if ctx_end_mo == 12 else 0)
            exp_mo = 1 if ctx_end_mo == 12 else ctx_end_mo + 1
            if int(hor.iloc[0]["year"]) != exp_yr or int(hor.iloc[0]["month"]) != exp_mo:
                continue
            hor_span = (
                (hor.iloc[-1]["year"] - hor.iloc[0]["year"]) * 12
                + hor.iloc[-1]["month"] - hor.iloc[0]["month"]
            )
            if hor_span != HORIZON_LEN - 1:
                continue
            scenarios.append({
                "merchant_id": mid,
                "context_data": ctx,
                "horizon_data": hor,
                "context_range": (
                    (int(ctx.iloc[0]["year"]), int(ctx.iloc[0]["month"])),
                    (ctx_end_yr, ctx_end_mo),
                ),
                "horizon_range": (
                    (exp_yr, exp_mo),
                    (int(hor.iloc[-1]["year"]), int(hor.iloc[-1]["month"])),
                ),
            })
    return scenarios


def merchant_split(scenarios, seed=42):
    rng = np.random.default_rng(seed)
    all_mids = sorted({s["merchant_id"] for s in scenarios})
    perm = rng.permutation(len(all_mids))
    n = len(all_mids)
    n_train = int(0.60 * n)
    n_val = int(0.20 * n)
    train_mids = {all_mids[i] for i in perm[:n_train]}
    val_mids = {all_mids[i] for i in perm[n_train : n_train + n_val]}
    test_mids = {all_mids[i] for i in perm[n_train + n_val :]}
    return (
        [s for s in scenarios if s["merchant_id"] in train_mids],
        [s for s in scenarios if s["merchant_id"] in val_mids],
        [s for s in scenarios if s["merchant_id"] in test_mids],
    )


def build_pool_caches(df, scenarios, cost_type_cols, target_col="avg_proc_cost_pct"):
    """Returns (flat_cache, knn_cache) keyed by (merchant_id, year, month)."""
    unique_keys = {
        (s["merchant_id"], int(s["context_data"].iloc[-1]["year"]),
         int(s["context_data"].iloc[-1]["month"]))
        for s in scenarios
    }
    print(f"  Building flat pool mean cache ({len(unique_keys):,} keys)...")
    flat_cache = {}
    for mid, yr, mo in unique_keys:
        snap = df[
            (df["merchant_id"] != mid)
            & ((df["year"] < yr) | ((df["year"] == yr) & (df["month"] <= mo)))
        ]
        flat_cache[(mid, yr, mo)] = float(snap[target_col].mean()) if len(snap) > 0 else 0.0

    knn_cache = dict(flat_cache)
    if not cost_type_cols:
        return flat_cache, knn_cache

    keys_by_date = defaultdict(list)
    for mid, yr, mo in unique_keys:
        keys_by_date[(yr, mo)].append(mid)

    n_dates = len(keys_by_date)
    print(f"  Building kNN pool mean cache ({len(unique_keys):,} keys, {n_dates} dates, k={KNN_K})...")

    for idx, ((yr, mo), query_mids) in enumerate(sorted(keys_by_date.items())):
        snap = df[(df["year"] < yr) | ((df["year"] == yr) & (df["month"] <= mo))]
        fp_all = snap.groupby("merchant_id")[cost_type_cols].mean()
        cost_all = snap.groupby("merchant_id")[target_col].mean()
        if len(fp_all) < KNN_K + 1:
            continue
        nn_date = NearestNeighbors(n_neighbors=min(KNN_K + 1, len(fp_all)), metric="cosine")
        nn_date.fit(fp_all.values)
        fp_index = fp_all.index.tolist()
        for mid in query_mids:
            ctx_row = df[(df["merchant_id"] == mid) & (df["year"] == yr) & (df["month"] == mo)]
            if len(ctx_row) == 0 or mid not in fp_all.index:
                continue
            _, raw_idx = nn_date.kneighbors(ctx_row[cost_type_cols].values)
            top_ids = [fp_index[i] for i in raw_idx[0] if fp_index[i] != mid][:KNN_K]
            if len(top_ids) == KNN_K:
                knn_cache[(mid, yr, mo)] = float(cost_all.loc[top_ids].mean())
        if (idx + 1) % 20 == 0:
            print(f"    {idx + 1}/{n_dates} dates processed...")

    print("  Pool caches complete.")
    return flat_cache, knn_cache


def adaptive_q(residuals, target=TARGET_COV):
    n = len(residuals)
    level = math.ceil((n + 1) * target) / n
    return float(np.quantile(residuals, level)) if level <= 1.0 else None


def effective_half_width(lo, hi):
    return float(np.mean((hi - lo) / 2.0))


def make_percentile_bins(ref_vals, apply_vals, pct_edges, min_count=MIN_POOL):
    pct_edges = np.array(pct_edges, dtype=float)
    edges = np.quantile(ref_vals, pct_edges)
    edges = np.maximum.accumulate(edges)
    edges = np.unique(edges)
    n_eff = len(edges) - 1
    if n_eff < 2:
        return None
    ref_bins = np.digitize(ref_vals, edges[1:-1], right=False)
    counts = np.array([(ref_bins == b).sum() for b in range(n_eff)])
    if counts.min() < min_count:
        return None
    apply_bins = np.digitize(apply_vals, edges[1:-1], right=False)
    return edges, ref_bins, apply_bins, n_eff, counts


def continuous_width_map(cal_scores_ref, apply_scores, cal_res, pct_edges, q_fallback):
    built = make_percentile_bins(cal_scores_ref, apply_scores, pct_edges, min_count=MIN_POOL)
    if built is None:
        return None
    edges, ref_bins, apply_bins, n_eff, counts = built
    q_vals = []
    for b in range(n_eff):
        res_b = cal_res[ref_bins == b].tolist()
        q_local = adaptive_q(res_b) if len(res_b) >= MIN_POOL else None
        q_vals.append(q_local if q_local is not None else q_fallback)
    q_vals = np.maximum.accumulate(np.array(q_vals, dtype=float))
    knot_x = 0.5 * (edges[:-1] + edges[1:])
    knot_x = np.maximum.accumulate(knot_x + np.arange(len(knot_x)) * 1e-9)
    hw_apply = np.interp(apply_scores, knot_x, q_vals, left=q_vals[0], right=q_vals[-1])
    return {
        "knot_x": knot_x, "q_vals": q_vals, "hw_apply": hw_apply,
    }


def generate_pool_ids(df, merchant_id, yr, mo):
    pool = df[
        (df["merchant_id"] != merchant_id)
        & ((df["year"] < yr) | ((df["year"] == yr) & (df["month"] <= mo)))
    ]
    return set(int(p) for p in pool["merchant_id"].unique())


def atomic_write(path, write_fn):
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=path.parent, prefix=".tmp_")
    try:
        os.close(fd)
        write_fn(Path(tmp))
        os.replace(tmp, path)
    except Exception:
        try:
            os.unlink(tmp)
        except OSError:
            pass
        raise


print("✓ Shared helpers loaded")

In [ ]:
# Cell 5: Load and validate data
print("Loading data...")
df_raw = pd.read_csv(DATA_PATH)
df_raw["year"] = df_raw["year"].astype(int)
df_raw["month"] = df_raw["month"].astype(int)

# Validate columns
required_base = {"merchant_id", "year", "month", "avg_proc_cost_pct"}
missing_base = required_base - set(df_raw.columns)
if missing_base:
    raise ValueError(f"Missing base columns: {missing_base}")

cost_missing = [c for c in COST_V2_REQUIRED if c not in df_raw.columns]
if cost_missing:
    print(f"⚠ Missing cost v2 columns (cost training will error): {cost_missing}")

tpv_missing = [c for c in TPV_V2_REQUIRED if c not in df_raw.columns]
if tpv_missing:
    print(f"⚠ Missing TPV v2 columns (TPV training will error): {tpv_missing}")

# Rolling window filter
today = date.today()
cutoff_year = today.year - WINDOW_YEARS
df = df_raw[
    (df_raw["year"] > cutoff_year)
    | ((df_raw["year"] == cutoff_year) & (df_raw["month"] >= today.month))
].copy()

print(f"✓ Loaded {len(df):,} rows, {df['merchant_id'].nunique():,} merchants")
print(f"  Window: {df['year'].min()}-{df['month'].min():02d} → {df['year'].max()}-{df['month'].max():02d}")
window_start = f"{int(df['year'].min())}-{int(df['month'].min()):02d}"
window_end = f"{int(df['year'].max())}-{int(df['month'].max()):02d}"

---
## Part A: AvgProcCost (M9) Forecast Training

Trains HuberRegressor models with 7 features + GBR risk models with 9 features.
Conformal intervals in **percentage space**.

In [ ]:
# Cell 6: Cost-specific feature builders

def cost_build_feature_matrix(scenarios, knn_cache):
    """(N, 7) feature matrix for cost forecast."""
    rows = []
    for s in scenarios:
        ctx = s["context_data"]
        vals = ctx["avg_proc_cost_pct"].values.astype(float)
        c_mean = float(np.mean(vals))
        c_std = float(np.std(vals))
        mom = float(vals[-1] - c_mean)
        key = (s["merchant_id"], int(ctx.iloc[-1]["year"]), int(ctx.iloc[-1]["month"]))
        p_mean = knn_cache[key]
        intra_std = float(ctx["std_proc_cost_pct"].fillna(0).mean())
        log_txn = float(np.log1p(ctx["transaction_count"].mean()))
        avg_med = float(np.mean(np.abs(vals - ctx["median_proc_cost_pct"].values)))
        rows.append([c_mean, c_std, mom, p_mean, intra_std, log_txn, avg_med])
    return np.array(rows, dtype=float)


def cost_build_risk_features(scenarios, flat_cache, knn_cache):
    """(N, 9) risk feature matrix for cost forecast."""
    rows = []
    for s in scenarios:
        ctx = s["context_data"]
        vals = ctx["avg_proc_cost_pct"].values.astype(float)
        c_mean = float(np.mean(vals))
        _denom = c_mean + _VOL_EPS
        intra_cov = float(ctx["std_proc_cost_pct"].fillna(0).mean()) / _denom
        mean_median_gap = float(np.mean(np.abs(vals - ctx["median_proc_cost_pct"].values))) / _denom
        log_txn_count = float(np.log1p(ctx["transaction_count"].mean()))
        ct_cols = [c for c in ctx.columns if c.startswith("cost_type_") and c.endswith("_pct")]
        ct_vals = ctx[ct_cols].mean().values if ct_cols else np.array([1.0])
        cost_type_hhi = float(np.sum(ct_vals ** 2))
        log_avg_txn_val = float(np.log1p(ctx["avg_transaction_value"].mean()))
        avg_txn_val = float(ctx["avg_transaction_value"].mean())
        txn_amount_cov = float(ctx["std_txn_amount"].fillna(0).mean()) / (avg_txn_val + _VOL_EPS)
        yr, mo = s["context_range"][1]
        mid = s["merchant_id"]
        flat_pm = float(flat_cache.get((mid, yr, mo), c_mean))
        knn_pm = float(knn_cache.get((mid, yr, mo), c_mean))
        pool_gap = abs(c_mean - flat_pm) / (flat_pm + _VOL_EPS)
        knn_gap = abs(c_mean - knn_pm) / (knn_pm + _VOL_EPS)
        ctx_cov = float(np.std(vals)) / _denom
        rows.append([
            intra_cov, mean_median_gap, log_txn_count,
            cost_type_hhi, log_avg_txn_val, txn_amount_cov,
            pool_gap, knn_gap, ctx_cov,
        ])
    return np.array(rows, dtype=float)


print("✓ Cost feature builders loaded")

In [ ]:
# Cell 7: Train AvgProcCost (M9) artifacts
%%time

print("="*60)
print(f"AvgProcCost Training  MCC={MCC}")
print("="*60)

# Build scenarios
print("\n[1/4] Building scenarios...")
cost_scenarios_by_ctx = {}
cost_all_scenarios = []
for ctx_len in SUPPORTED_CONTEXT_LENS:
    scens = build_scenarios(df, ctx_len)
    cost_scenarios_by_ctx[ctx_len] = scens
    cost_all_scenarios.extend(scens)
    print(f"  ctx={ctx_len}: {len(scens):,} scenarios from {len({s['merchant_id'] for s in scens}):,} merchants")

# Build pool caches
print("\n[2/4] Building pool caches...")
present_cost_cols = [c for c in COST_TYPE_COLS if c in df.columns]
cost_flat_cache, cost_knn_cache = build_pool_caches(
    df, cost_all_scenarios, present_cost_cols, target_col="avg_proc_cost_pct"
)

# Train per context length
print("\n[3/4] Training per context length...")
for ctx_len in SUPPORTED_CONTEXT_LENS:
    scens = cost_scenarios_by_ctx[ctx_len]
    if len(scens) < 50:
        print(f"\n  SKIP ctx_len={ctx_len}: only {len(scens)} scenarios")
        continue
    t0 = time.monotonic()
    print(f"\n{'─'*60}")
    print(f"  ctx_len={ctx_len}")
    print(f"{'─'*60}")

    train_s, val_s, test_s = merchant_split(scens)
    print(f"  train={len(train_s):,}  val={len(val_s):,}  test={len(test_s):,}")

    ci_years = sorted({int(s["horizon_data"].iloc[0]["year"]) for s in (train_s + val_s + test_s)})
    if len(ci_years) < 2:
        print(f"  SKIP: fewer than 2 horizon years"); continue
    cal_year, test_year = ci_years[-2], ci_years[-1]

    train_ci = [s for s in train_s if int(s["horizon_data"].iloc[0]["year"]) < cal_year]
    cal_ci = [s for s in val_s if int(s["horizon_data"].iloc[0]["year"]) == cal_year]
    test_ci = [s for s in test_s if int(s["horizon_data"].iloc[0]["year"]) == test_year]
    if not train_ci or not cal_ci:
        print(f"  SKIP: empty train_ci or cal_ci"); continue
    print(f"  cal_year={cal_year}  test_year={test_year}")
    print(f"  train_ci={len(train_ci):,}  cal_ci={len(cal_ci):,}  test_ci={len(test_ci):,}")

    # Fit models
    y_tr = np.array([s["horizon_data"]["avg_proc_cost_pct"].values for s in train_ci])
    y_cal = np.array([s["horizon_data"]["avg_proc_cost_pct"].values for s in cal_ci])
    X_tr_raw = cost_build_feature_matrix(train_ci, cost_knn_cache)
    X_cal_raw = cost_build_feature_matrix(cal_ci, cost_knn_cache)
    sw = 1.0 / (X_tr_raw[:, 3] + 1.0)

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr_raw)
    X_cal = scaler.transform(X_cal_raw)

    models = []
    cal_preds = np.zeros_like(y_cal, dtype=float)
    for h in range(HORIZON_LEN):
        m = HuberRegressor(epsilon=1.35, max_iter=500)
        m.fit(X_tr, y_tr[:, h], sample_weight=sw)
        cal_preds[:, h] = m.predict(X_cal)
        models.append(m)
        print(f"  h={h+1}  coefs={m.coef_.round(4)}")

    # Calibration
    cal_max_res = np.abs(y_cal - cal_preds).max(axis=1)
    global_q90 = float(np.quantile(cal_max_res, TARGET_COV))
    print(f"  global_q90 = ±{global_q90:.4f}")

    cal_merchant_ids = np.array([s["merchant_id"] for s in cal_ci])
    cal_residuals = defaultdict(list)
    for i, res in enumerate(cal_max_res):
        cal_residuals[int(cal_merchant_ids[i])].append(float(res))
    cal_residuals = dict(cal_residuals)

    # Cross-fitted GBR risk models
    print("  Training GBR risk models...")
    train_feat = cost_build_risk_features(train_ci, cost_flat_cache, cost_knn_cache)
    train_years = np.array([int(s["horizon_data"].iloc[0]["year"]) for s in train_ci])
    fold_cuts = sorted(set(train_years.tolist()))[1:]
    train_cf_res = np.full((len(train_ci), HORIZON_LEN), np.nan)

    for cut in fold_cuts:
        cf_tr_mask = train_years < cut
        cf_te_mask = train_years == cut
        if cf_tr_mask.sum() == 0 or cf_te_mask.sum() == 0:
            continue
        cf_tr = [train_ci[j] for j in np.where(cf_tr_mask)[0]]
        cf_te = [train_ci[j] for j in np.where(cf_te_mask)[0]]
        y_cf_tr = np.array([s["horizon_data"]["avg_proc_cost_pct"].values for s in cf_tr])
        y_cf_te = np.array([s["horizon_data"]["avg_proc_cost_pct"].values for s in cf_te])
        X_cf_tr = cost_build_feature_matrix(cf_tr, cost_knn_cache)
        X_cf_te = cost_build_feature_matrix(cf_te, cost_knn_cache)
        sw_cf = 1.0 / (X_cf_tr[:, 3] + 1.0)
        sc_cf = StandardScaler()
        Xcf_tr = sc_cf.fit_transform(X_cf_tr)
        Xcf_te = sc_cf.transform(X_cf_te)
        preds_cf = np.zeros_like(y_cf_te, dtype=float)
        for h in range(HORIZON_LEN):
            mcf = HuberRegressor(epsilon=1.35, max_iter=500)
            mcf.fit(Xcf_tr, y_cf_tr[:, h], sample_weight=sw_cf)
            preds_cf[:, h] = mcf.predict(Xcf_te)
        train_cf_res[cf_te_mask] = np.abs(y_cf_te - preds_cf)

    cf_valid = np.isfinite(train_cf_res).all(axis=1)
    risk_models = []
    for h in range(HORIZON_LEN):
        gbr = GradientBoostingRegressor(
            loss="squared_error", n_estimators=GBR_N_ESTIMATORS,
            learning_rate=GBR_LEARNING_RATE, max_depth=GBR_MAX_DEPTH,
            min_samples_leaf=GBR_MIN_SAMPLES_LEAF, subsample=GBR_SUBSAMPLE,
            random_state=GBR_RANDOM_STATE,
        )
        gbr.fit(train_feat[cf_valid], np.log1p(train_cf_res[cf_valid, h]))
        risk_models.append(gbr)

    # Stratification scheme selection
    strat_enabled = False
    strat_knot_x = strat_q_vals = strat_scheme = None

    if cf_valid.sum() >= 50:
        print("  Selecting stratification scheme...")
        cal_feat = cost_build_risk_features(cal_ci, cost_flat_cache, cost_knn_cache)
        cal_risk = np.max(np.column_stack([m.predict(cal_feat) for m in risk_models]), axis=1)
        cal_mids_arr = np.array(sorted(set(cal_merchant_ids.tolist())))
        rng = np.random.default_rng(GBR_RANDOM_STATE)
        perm = rng.permutation(cal_mids_arr)
        cut = min(max(1, int(round(len(perm) * 0.70))), len(perm) - 1)
        sel_mids = set(perm[:cut].tolist())
        sel_mask = np.isin(cal_merchant_ids, list(sel_mids))
        eval_mask = ~sel_mask

        if eval_mask.sum() > 0:
            y_eval = y_cal[eval_mask]
            preds_eval = cal_preds[eval_mask]
            eval_ci_list = [cal_ci[j] for j in np.where(eval_mask)[0]]
            merchant_sel_res = defaultdict(list)
            for mid_val, res in zip(cal_merchant_ids[sel_mask], cal_max_res[sel_mask]):
                merchant_sel_res[int(mid_val)].append(float(res))
            global_q_sel = adaptive_q(cal_max_res[sel_mask].tolist()) or global_q90

            hw_eval_pool = np.zeros(eval_mask.sum(), dtype=float)
            for i, s in enumerate(eval_ci_list):
                yr, mo = s["context_range"][1]
                peer_ids = generate_pool_ids(df, int(s["merchant_id"]), yr, mo)
                peer_res = [r for pid in peer_ids for r in merchant_sel_res.get(pid, [])]
                q = adaptive_q(peer_res) if len(peer_res) >= MIN_POOL else None
                hw_eval_pool[i] = q if q is not None else global_q_sel

            lo_ep = np.clip(preds_eval - hw_eval_pool[:, None], 0, None)
            hi_ep = preds_eval + hw_eval_pool[:, None]
            baseline_eval_hw = effective_half_width(lo_ep, hi_ep)
            _min_gain = max(VOL_MIN_GAIN_ABS, VOL_MIN_GAIN_REL * baseline_eval_hw)

            sel_risk = cal_risk[sel_mask]
            eval_risk = cal_risk[eval_mask]
            scheme_results = []
            for _name, _pct in VOL_BUCKET_SCHEMES.items():
                mapped = continuous_width_map(sel_risk, eval_risk, cal_max_res[sel_mask], np.array(_pct), global_q_sel)
                if mapped is None: continue
                hw = mapped["hw_apply"]
                lo_ = np.clip(preds_eval - hw[:, None], 0, None)
                hi_ = preds_eval + hw[:, None]
                in_ = (y_eval >= lo_) & (y_eval <= hi_)
                avg_hw = effective_half_width(lo_, hi_)
                joint_cov = float(np.mean(in_.all(axis=1)))
                scheme_results.append({"name": _name, "pct_edges": np.array(_pct),
                    "avg_hw": avg_hw, "joint_cov": joint_cov, "gain": baseline_eval_hw - avg_hw})

            feasible = [r for r in scheme_results if r["joint_cov"] >= TARGET_COV and r["gain"] >= _min_gain]
            if feasible:
                best = min(feasible, key=lambda r: r["avg_hw"])
                mapped_full = continuous_width_map(cal_risk, cal_risk, cal_max_res, best["pct_edges"], global_q90)
                if mapped_full is not None:
                    strat_enabled = True
                    strat_scheme = best["name"]
                    strat_knot_x = mapped_full["knot_x"]
                    strat_q_vals = mapped_full["q_vals"]
                    print(f"  ✓ Stratification PASSED: {strat_scheme}")
                else:
                    print("  ✗ Stratification failed on full cal set")
            else:
                print(f"  ✗ No feasible scheme (tested {len(scheme_results)})")

    # Write artifacts
    artifact_dir = M9_ARTIFACTS / str(MCC) / str(ctx_len)
    artifact_dir.mkdir(parents=True, exist_ok=True)
    atomic_write(artifact_dir / "models.pkl", lambda p: joblib.dump(models, p))
    atomic_write(artifact_dir / "scaler.pkl", lambda p: joblib.dump(scaler, p))
    atomic_write(artifact_dir / "cal_residuals.pkl", lambda p: joblib.dump(cal_residuals, p))
    atomic_write(artifact_dir / "global_q90.pkl", lambda p: joblib.dump(global_q90, p))
    atomic_write(artifact_dir / "risk_models.pkl", lambda p: joblib.dump(risk_models, p))
    if strat_enabled and strat_knot_x is not None:
        atomic_write(artifact_dir / "strat_knot_x.pkl", lambda p: joblib.dump(strat_knot_x, p))
        atomic_write(artifact_dir / "strat_q_vals.pkl", lambda p: joblib.dump(strat_q_vals, p))

    snapshot = {
        "trained_at": datetime.now(timezone.utc).isoformat(),
        "mcc": MCC, "context_len": ctx_len, "horizon_len": HORIZON_LEN,
        "window_start": window_start, "window_end": window_end,
        "n_train_ci": len(train_ci), "n_cal_ci": len(cal_ci), "n_test_ci": len(test_ci),
        "cal_year": cal_year, "global_q90": global_q90,
        "knn_k": KNN_K, "target_cov": TARGET_COV,
        "strat_enabled": strat_enabled, "strat_scheme": strat_scheme,
        "n_model_features": 7, "n_risk_features": 9,
    }
    atomic_write(artifact_dir / "config_snapshot.json",
                 lambda p: p.write_text(json.dumps(snapshot, indent=2)))
    print(f"  ✓ Artifacts → {artifact_dir}  [{time.monotonic()-t0:.1f}s]")

print("\n" + "="*60)
print("✓ AvgProcCost training complete")
print("="*60)

---
## Part B: TPV Forecast Training

Trains HuberRegressor models with 11 features in **log-space** + GBR risk models with 11 features.
Conformal intervals in **dollar space** (expm1 back-transform).

In [ ]:
# Cell 8: TPV-specific feature builders

def tpv_build_feature_matrix(scenarios, knn_cache):
    """(N, 11) feature matrix for TPV forecast (log-space)."""
    rows = []
    for s in scenarios:
        ctx = s["context_data"]
        vals = ctx[LOG_TARGET].values.astype(float)
        c_mean = float(np.mean(vals))
        c_std = float(np.std(vals))
        mom = float(vals[-1] - c_mean)
        key = (s["merchant_id"], int(ctx.iloc[-1]["year"]), int(ctx.iloc[-1]["month"]))
        p_mean = knn_cache[key]
        txn_amount_std = float(ctx["std_txn_amount"].fillna(0).mean())
        log_txn = float(np.log1p(ctx["transaction_count"].mean()))
        avg_median_gap = float(np.mean(np.abs(
            ctx["avg_transaction_value"].values - ctx["median_txn_amount"].values
        )))
        last_month = float(vals[-1])
        log_avg_txn_val = float(np.log1p(ctx["avg_transaction_value"].mean()))
        tc_vals = np.log1p(ctx["transaction_count"].values.astype(float))
        atv_vals = np.log1p(ctx["avg_transaction_value"].values.astype(float))
        mom_tc = float(tc_vals[-1] - np.mean(tc_vals))
        mom_atv = float(atv_vals[-1] - np.mean(atv_vals))
        rows.append([c_mean, c_std, mom, p_mean, txn_amount_std, log_txn,
                     avg_median_gap, last_month, log_avg_txn_val, mom_tc, mom_atv])
    return np.array(rows, dtype=float)


def tpv_build_risk_features(scenarios, flat_cache, knn_cache):
    """(N, 11) risk feature matrix for TPV forecast."""
    rows = []
    for s in scenarios:
        ctx = s["context_data"]
        vals = ctx[LOG_TARGET].values.astype(float)
        c_mean = float(np.mean(vals))
        _denom = c_mean + _VOL_EPS
        avg_txn_val = float(ctx["avg_transaction_value"].mean())
        intra_txn_cov = float(ctx["std_txn_amount"].fillna(0).mean()) / (avg_txn_val + _VOL_EPS)
        avg_median_gap = float(np.mean(np.abs(
            ctx["avg_transaction_value"].values - ctx["median_txn_amount"].values
        ))) / (avg_txn_val + _VOL_EPS)
        log_txn_count = float(np.log1p(ctx["transaction_count"].mean()))
        ct_cols = [c for c in ctx.columns if c.startswith("cost_type_") and c.endswith("_pct")]
        ct_vals = ctx[ct_cols].mean().values if ct_cols else np.array([1.0])
        cost_type_hhi = float(np.sum(ct_vals ** 2))
        log_avg_txn_val = float(np.log1p(avg_txn_val))
        txn_amount_cov = float(ctx["std_txn_amount"].fillna(0).mean()) / (avg_txn_val + _VOL_EPS)
        yr, mo = s["context_range"][1]
        mid = s["merchant_id"]
        flat_pm = float(flat_cache.get((mid, yr, mo), c_mean))
        knn_pm = float(knn_cache.get((mid, yr, mo), c_mean))
        pool_gap = abs(c_mean - flat_pm) / (flat_pm + _VOL_EPS)
        knn_gap = abs(c_mean - knn_pm) / (knn_pm + _VOL_EPS)
        ctx_cov = float(np.std(vals)) / _denom
        tc_vals_arr = np.log1p(ctx["transaction_count"].values.astype(float))
        atv_vals_arr = np.log1p(ctx["avg_transaction_value"].values.astype(float))
        tc_cov = float(np.std(tc_vals_arr)) / (float(np.mean(tc_vals_arr)) + _VOL_EPS)
        atv_cov = float(np.std(atv_vals_arr)) / (float(np.mean(atv_vals_arr)) + _VOL_EPS)
        rows.append([
            intra_txn_cov, avg_median_gap, log_txn_count,
            cost_type_hhi, log_avg_txn_val, txn_amount_cov,
            pool_gap, knn_gap, ctx_cov, tc_cov, atv_cov,
        ])
    return np.array(rows, dtype=float)


print("✓ TPV feature builders loaded")

In [ ]:
# Cell 9: Train TPV forecast artifacts
%%time

# Prepare log target
df[LOG_TARGET] = np.log1p(df[TARGET_COL].astype(float))

print("="*60)
print(f"TPV Forecast Training  MCC={MCC}")
print("="*60)

# Build scenarios
print("\n[1/4] Building scenarios...")
tpv_scenarios_by_ctx = {}
tpv_all_scenarios = []
for ctx_len in SUPPORTED_CONTEXT_LENS:
    scens = build_scenarios(df, ctx_len)
    tpv_scenarios_by_ctx[ctx_len] = scens
    tpv_all_scenarios.extend(scens)
    print(f"  ctx={ctx_len}: {len(scens):,} scenarios from {len({s['merchant_id'] for s in scens}):,} merchants")

# Build pool caches (log-space for TPV)
print("\n[2/4] Building pool caches (log TPV space)...")
tpv_flat_cache, tpv_knn_cache = build_pool_caches(
    df, tpv_all_scenarios, present_cost_cols, target_col=LOG_TARGET
)

# Train per context length
print("\n[3/4] Training per context length...")
for ctx_len in SUPPORTED_CONTEXT_LENS:
    scens = tpv_scenarios_by_ctx[ctx_len]
    if len(scens) < 50:
        print(f"\n  SKIP ctx_len={ctx_len}: only {len(scens)} scenarios")
        continue
    t0 = time.monotonic()
    print(f"\n{'─'*60}")
    print(f"  ctx_len={ctx_len}")
    print(f"{'─'*60}")

    train_s, val_s, test_s = merchant_split(scens)
    ci_years = sorted({int(s["horizon_data"].iloc[0]["year"]) for s in (train_s + val_s + test_s)})
    if len(ci_years) < 2:
        print(f"  SKIP: fewer than 2 horizon years"); continue
    cal_year, test_year = ci_years[-2], ci_years[-1]

    train_ci = [s for s in train_s if int(s["horizon_data"].iloc[0]["year"]) < cal_year]
    cal_ci = [s for s in val_s if int(s["horizon_data"].iloc[0]["year"]) == cal_year]
    test_ci = [s for s in test_s if int(s["horizon_data"].iloc[0]["year"]) == test_year]
    if not train_ci or not cal_ci:
        print(f"  SKIP: empty train_ci or cal_ci"); continue
    print(f"  cal_year={cal_year}  test_year={test_year}")
    print(f"  train_ci={len(train_ci):,}  cal_ci={len(cal_ci):,}  test_ci={len(test_ci):,}")

    # Fit models (log-space, dollar-weighted)
    y_tr_log = np.array([s["horizon_data"][LOG_TARGET].values for s in train_ci])
    y_cal_log = np.array([s["horizon_data"][LOG_TARGET].values for s in cal_ci])
    y_cal_dollar = np.array([s["horizon_data"][TARGET_COL].values for s in cal_ci])

    X_tr_raw = tpv_build_feature_matrix(train_ci, tpv_knn_cache)
    X_cal_raw = tpv_build_feature_matrix(cal_ci, tpv_knn_cache)

    sw = np.array([
        (1.0 / np.log1p(s["context_data"]["transaction_count"].mean()))
        * np.expm1(np.mean(s["context_data"][LOG_TARGET].values))
        for s in train_ci
    ], dtype=float)

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr_raw)
    X_cal = scaler.transform(X_cal_raw)

    models = []
    cal_preds_log = np.zeros_like(y_cal_log, dtype=float)
    for h in range(HORIZON_LEN):
        m = HuberRegressor(epsilon=1.35, max_iter=500)
        m.fit(X_tr, y_tr_log[:, h], sample_weight=sw)
        cal_preds_log[:, h] = m.predict(X_cal)
        models.append(m)
        print(f"  h={h+1}  coefs={m.coef_.round(4)}")

    # Dollar-space calibration
    cal_preds_dollar = np.expm1(cal_preds_log)
    cal_res_dollar = np.abs(y_cal_dollar - cal_preds_dollar)
    cal_max_res = cal_res_dollar.max(axis=1)
    global_q90 = float(np.quantile(cal_max_res, TARGET_COV))
    print(f"  global_q90 = ±${global_q90:.2f}")

    cal_merchant_ids = np.array([s["merchant_id"] for s in cal_ci])
    cal_residuals = defaultdict(list)
    for i, res in enumerate(cal_max_res):
        cal_residuals[int(cal_merchant_ids[i])].append(float(res))
    cal_residuals = dict(cal_residuals)

    # Cross-fitted GBR risk models
    print("  Training GBR risk models...")
    train_feat = tpv_build_risk_features(train_ci, tpv_flat_cache, tpv_knn_cache)
    train_years = np.array([int(s["horizon_data"].iloc[0]["year"]) for s in train_ci])
    fold_cuts = sorted(set(train_years.tolist()))[1:]
    train_cf_res = np.full((len(train_ci), HORIZON_LEN), np.nan)

    for cut in fold_cuts:
        cf_tr_mask = train_years < cut
        cf_te_mask = train_years == cut
        if cf_tr_mask.sum() == 0 or cf_te_mask.sum() == 0:
            continue
        cf_tr = [train_ci[j] for j in np.where(cf_tr_mask)[0]]
        cf_te = [train_ci[j] for j in np.where(cf_te_mask)[0]]
        y_cf_tr_log = np.array([s["horizon_data"][LOG_TARGET].values for s in cf_tr])
        y_cf_te_log = np.array([s["horizon_data"][LOG_TARGET].values for s in cf_te])
        y_cf_te_dol = np.array([s["horizon_data"][TARGET_COL].values for s in cf_te])
        X_cf_tr = tpv_build_feature_matrix(cf_tr, tpv_knn_cache)
        X_cf_te = tpv_build_feature_matrix(cf_te, tpv_knn_cache)
        sw_cf = np.array([
            (1.0 / np.log1p(s["context_data"]["transaction_count"].mean()))
            * np.expm1(np.mean(s["context_data"][LOG_TARGET].values))
            for s in cf_tr
        ], dtype=float)
        sc_cf = StandardScaler()
        Xcf_tr = sc_cf.fit_transform(X_cf_tr)
        Xcf_te = sc_cf.transform(X_cf_te)
        preds_cf_log = np.zeros_like(y_cf_te_log, dtype=float)
        for h in range(HORIZON_LEN):
            mcf = HuberRegressor(epsilon=1.35, max_iter=500)
            mcf.fit(Xcf_tr, y_cf_tr_log[:, h], sample_weight=sw_cf)
            preds_cf_log[:, h] = mcf.predict(Xcf_te)
        preds_cf_dollar = np.expm1(preds_cf_log)
        train_cf_res[cf_te_mask] = np.abs(y_cf_te_dol - preds_cf_dollar)

    cf_valid = np.isfinite(train_cf_res).all(axis=1)
    risk_models = []
    for h in range(HORIZON_LEN):
        gbr = GradientBoostingRegressor(
            loss="squared_error", n_estimators=GBR_N_ESTIMATORS,
            learning_rate=GBR_LEARNING_RATE, max_depth=GBR_MAX_DEPTH,
            min_samples_leaf=GBR_MIN_SAMPLES_LEAF, subsample=GBR_SUBSAMPLE,
            random_state=GBR_RANDOM_STATE,
        )
        gbr.fit(train_feat[cf_valid], np.log1p(train_cf_res[cf_valid, h]))
        risk_models.append(gbr)

    # Stratification
    strat_enabled = False
    strat_knot_x = strat_q_vals = strat_scheme = None

    if cf_valid.sum() >= 50:
        print("  Selecting stratification scheme...")
        cal_feat = tpv_build_risk_features(cal_ci, tpv_flat_cache, tpv_knn_cache)
        cal_risk = np.max(np.column_stack([m.predict(cal_feat) for m in risk_models]), axis=1)
        cal_mids_arr = np.array(sorted(set(cal_merchant_ids.tolist())))
        rng = np.random.default_rng(GBR_RANDOM_STATE)
        perm = rng.permutation(cal_mids_arr)
        cut_idx = min(max(1, int(round(len(perm) * 0.70))), len(perm) - 1)
        sel_mids = set(perm[:cut_idx].tolist())
        sel_mask = np.isin(cal_merchant_ids, list(sel_mids))
        eval_mask = ~sel_mask

        if eval_mask.sum() > 0:
            y_eval_dollar = y_cal_dollar[eval_mask]
            preds_eval_dol = cal_preds_dollar[eval_mask]
            eval_ci_list = [cal_ci[j] for j in np.where(eval_mask)[0]]
            merchant_sel_res = defaultdict(list)
            for mid_val, res in zip(cal_merchant_ids[sel_mask], cal_max_res[sel_mask]):
                merchant_sel_res[int(mid_val)].append(float(res))
            global_q_sel = adaptive_q(cal_max_res[sel_mask].tolist()) or global_q90

            hw_eval_pool = np.zeros(eval_mask.sum(), dtype=float)
            for i, s in enumerate(eval_ci_list):
                yr, mo = s["context_range"][1]
                peer_ids = generate_pool_ids(df, int(s["merchant_id"]), yr, mo)
                peer_res = [r for pid in peer_ids for r in merchant_sel_res.get(pid, [])]
                q = adaptive_q(peer_res) if len(peer_res) >= MIN_POOL else None
                hw_eval_pool[i] = q if q is not None else global_q_sel

            baseline_eval_hw = float(np.mean(hw_eval_pool))
            _min_gain = max(VOL_MIN_GAIN_ABS, VOL_MIN_GAIN_REL * baseline_eval_hw)
            sel_risk = cal_risk[sel_mask]
            eval_risk = cal_risk[eval_mask]

            scheme_results = []
            for _name, _pct in VOL_BUCKET_SCHEMES.items():
                mapped = continuous_width_map(sel_risk, eval_risk, cal_max_res[sel_mask], np.array(_pct), global_q_sel)
                if mapped is None: continue
                hw = mapped["hw_apply"]
                lo_ = np.clip(preds_eval_dol - hw[:, None], 0, None)
                hi_ = preds_eval_dol + hw[:, None]
                in_ = (y_eval_dollar >= lo_) & (y_eval_dollar <= hi_)
                avg_hw = float(np.mean(hw))
                joint_cov = float(np.mean(in_.all(axis=1)))
                scheme_results.append({"name": _name, "pct_edges": np.array(_pct),
                    "avg_hw": avg_hw, "joint_cov": joint_cov, "gain": baseline_eval_hw - avg_hw})

            feasible = [r for r in scheme_results if r["joint_cov"] >= TARGET_COV and r["gain"] >= _min_gain]
            if feasible:
                best = min(feasible, key=lambda r: r["avg_hw"])
                mapped_full = continuous_width_map(cal_risk, cal_risk, cal_max_res, best["pct_edges"], global_q90)
                if mapped_full is not None:
                    strat_enabled = True
                    strat_scheme = best["name"]
                    strat_knot_x = mapped_full["knot_x"]
                    strat_q_vals = mapped_full["q_vals"]
                    print(f"  ✓ Stratification PASSED: {strat_scheme}")
                else:
                    print("  ✗ Stratification failed on full cal set")
            else:
                print(f"  ✗ No feasible scheme (tested {len(scheme_results)})")

    # Write artifacts
    artifact_dir = TPV_ARTIFACTS / str(MCC) / str(ctx_len)
    artifact_dir.mkdir(parents=True, exist_ok=True)
    atomic_write(artifact_dir / "models.pkl", lambda p: joblib.dump(models, p))
    atomic_write(artifact_dir / "scaler.pkl", lambda p: joblib.dump(scaler, p))
    atomic_write(artifact_dir / "cal_residuals.pkl", lambda p: joblib.dump(cal_residuals, p))
    atomic_write(artifact_dir / "global_q90.pkl", lambda p: joblib.dump(global_q90, p))
    atomic_write(artifact_dir / "risk_models.pkl", lambda p: joblib.dump(risk_models, p))
    if strat_enabled and strat_knot_x is not None:
        atomic_write(artifact_dir / "strat_knot_x.pkl", lambda p: joblib.dump(strat_knot_x, p))
        atomic_write(artifact_dir / "strat_q_vals.pkl", lambda p: joblib.dump(strat_q_vals, p))

    snapshot = {
        "trained_at": datetime.now(timezone.utc).isoformat(),
        "mcc": MCC, "context_len": ctx_len, "horizon_len": HORIZON_LEN,
        "window_start": window_start, "window_end": window_end,
        "n_train_ci": len(train_ci), "n_cal_ci": len(cal_ci), "n_test_ci": len(test_ci),
        "cal_year": cal_year, "global_q90_dollars": global_q90,
        "knn_k": KNN_K, "target_cov": TARGET_COV,
        "strat_enabled": strat_enabled, "strat_scheme": strat_scheme,
        "n_model_features": 11, "n_risk_features": 11,
        "conformal_space": "dollar", "bias_correction": "none",
        "sample_weighting": "dollar_weighted",
    }
    atomic_write(artifact_dir / "config_snapshot.json",
                 lambda p: p.write_text(json.dumps(snapshot, indent=2)))
    print(f"  ✓ Artifacts → {artifact_dir}  [{time.monotonic()-t0:.1f}s]")

print("\n" + "="*60)
print("✓ TPV Forecast training complete")
print("="*60)

---
## Download Artifacts

After training, download the zipped artifact folders.

**Integration:**
- Unzip `m9_artifacts.zip` → copy contents to `ml_service/artifacts/m9/`
- Unzip `tpv_artifacts.zip` → copy contents to `ml_service/artifacts/tpv/`

Each zip contains: `{mcc}/{ctx_len}/` with models.pkl, scaler.pkl, cal_residuals.pkl, global_q90.pkl, risk_models.pkl, and optionally strat_knot_x.pkl + strat_q_vals.pkl + config_snapshot.json.

In [ ]:
# Cell 10: Zip and download artifacts
import shutil

# List what was produced
print("M9 (AvgProcCost) artifacts:")
for p in sorted(M9_ARTIFACTS.rglob("*")):
    if p.is_file():
        print(f"  {p}  ({p.stat().st_size:,} bytes)")

print("\nTPV artifacts:")
for p in sorted(TPV_ARTIFACTS.rglob("*")):
    if p.is_file():
        print(f"  {p}  ({p.stat().st_size:,} bytes)")

# Zip
shutil.make_archive("m9_artifacts", "zip", ".", str(M9_ARTIFACTS))
shutil.make_archive("tpv_artifacts", "zip", ".", str(TPV_ARTIFACTS))

print("\n✓ Zipped. Downloading...")
files.download("m9_artifacts.zip")
files.download("tpv_artifacts.zip")